In [1]:
import numpy as np
from math import ceil

In [2]:
w=[0.04, 0.20, 0.40, 0.28, 0.10, 0.16, 0.03, 0.08]
v=[0.10, 0.14, 0.24, 0.32, 0.28, 0.24, 0.18, 0.14]
q=[30, 20, 12, 23, 15, 30, 40, 25]
pi1=[1/10, 1/5, 1/2, 1/3, 1/3, 1/4, 1/5, 1/7]

In [11]:
a=[]
for i in range(8):
    a.append(min(int(1/w[i]), int(1/v[i])))
A=np.diag(a)
neg_I = -1 * np.eye(8)
A = np.hstack([A, neg_I])
A

array([[10.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -1., -0., -0., -0., -0.,
        -0., -0., -0.],
       [ 0.,  5.,  0.,  0.,  0.,  0.,  0.,  0., -0., -1., -0., -0., -0.,
        -0., -0., -0.],
       [ 0.,  0.,  2.,  0.,  0.,  0.,  0.,  0., -0., -0., -1., -0., -0.,
        -0., -0., -0.],
       [ 0.,  0.,  0.,  3.,  0.,  0.,  0.,  0., -0., -0., -0., -1., -0.,
        -0., -0., -0.],
       [ 0.,  0.,  0.,  0.,  3.,  0.,  0.,  0., -0., -0., -0., -0., -1.,
        -0., -0., -0.],
       [ 0.,  0.,  0.,  0.,  0.,  4.,  0.,  0., -0., -0., -0., -0., -0.,
        -1., -0., -0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  5.,  0., -0., -0., -0., -0., -0.,
        -0., -1., -0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  7., -0., -0., -0., -0., -0.,
        -0., -0., -1.]])

In [14]:
c1 = [1]*8 + [0]*8
B = A[:, :8]
cB=[1]*8
b1=np.linalg.inv(B)@q
A=np.linalg.inv(B)@A
sigma1=c1-cB@A
sigma1


array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.1       , 0.2       ,
       0.5       , 0.33333333, 0.33333333, 0.25      , 0.2       ,
       0.14285714])

In [3]:
wpi1=[w[i]/pi1[i] for i in range(8)]
wpi1

[0.39999999999999997,
 1.0,
 0.8,
 0.8400000000000001,
 0.30000000000000004,
 0.64,
 0.15,
 0.56]

In [4]:
vpi1=[v[i]/pi1[i] for i in range(8)]
vpi1

[1.0,
 0.7000000000000001,
 0.48,
 0.9600000000000001,
 0.8400000000000001,
 0.96,
 0.8999999999999999,
 0.9800000000000001]

找同时满足w和v约束代价最小的，依次为3、5、7、4、6、8、2、1，依次赋予可取的最大整数值，得到新的运输方案a9=[0,0,2,0,1,0,1,0]，代入上一步单纯形的列为B^(-1)*a9

In [17]:
a9=[0,0,2,0,1,0,1,0]
a9=np.linalg.inv(B)@a9
A=np.hstack([A, a9.reshape(-1,1)])
A

array([[ 1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        , -0.1       ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        , -0.2       ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        -0.5       ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  1.        ],
       [ 0.        ,  0.        ,  0.        ,  1.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        , -0.33333333,  0.        ,  0.        ,  0.        ,
         0.        

In [26]:
# c1=[1]*8+[0]*8
# c1.append(1)
sigma2=c1-cB@A
# print(sigma2) #证实了a9需进基
theta=b1/a9
print(theta)

[inf inf  6. inf 15. inf 40. inf]


/var/folders/_9/bwrp540d21j6tn6bbmtzrxww0000gn/T/ipykernel_59585/1276806201.py:5: RuntimeWarning: divide by zero encountered in divide
  theta=b1/a9


取最小theta对应的变量出基，即x_3

## 列生成法完整实现（固定初始基 + 最优表加列）

按你给的逻辑实现：

1. 给定初始主问题并选初始基（记为 `basis0`），在其基础上得到当前最优单纯形表。  
2. 当前基对应目标系数为 `cB`。取“初始基变量在当前最优表中的列矩阵”为 `B逆`。  
3. 乘子（对偶）按 `pi = cB @ B逆` 计算。  
4. 定价在初始约束下找新列；把新列转成初始基坐标 `p_j` 后，计算检验数。  
5. 若改进，则将 `B逆 @ p_j` 作为新列加入**当前最优表**（尚未入基）。  
6. 对这个“已加列但未重优化”的表继续做单纯形优化，进入下一轮。

In [28]:
def simplex_optimize_tableau(A_tab, b_tab, c, basis, max_iter=5000, tol=1e-10):
    """
    在当前单纯形表上继续优化（最小化）：
    - A_tab: 当前表中的约束系数矩阵（行是约束，列是变量）
    - b_tab: 当前表 RHS（即当前基变量值）
    - c: 全变量目标系数
    - basis: 当前基索引
    返回 (obj, A_tab, b_tab, basis, cB, sigma)
    """
    m, n = A_tab.shape
    c = np.asarray(c, dtype=float)
    b_tab = np.asarray(b_tab, dtype=float)

    for _ in range(max_iter):
        cB = c[basis]
        sigma = c - cB @ A_tab
        non_basis = [j for j in range(n) if j not in basis]
        sigma_N = sigma[non_basis]

        # 最小化：全部非基检验数 >= 0 时最优
        if np.all(sigma_N >= -tol):
            obj = float(cB @ b_tab)
            return obj, A_tab, b_tab, basis, cB, sigma

        # 入基：最负检验数
        j_enter = non_basis[int(np.argmin(sigma_N))]
        col = A_tab[:, j_enter]

        # 无界判断
        if np.all(col <= tol):
            return None, A_tab, b_tab, basis, cB, sigma

        theta = np.where(col > tol, b_tab / col, np.inf)
        i_leave = int(np.argmin(theta))
        if not np.isfinite(theta[i_leave]):
            return None, A_tab, b_tab, basis, cB, sigma

        # 枢轴操作（对整张表做行变换）
        pivot = A_tab[i_leave, j_enter]
        A_tab[i_leave, :] = A_tab[i_leave, :] / pivot
        b_tab[i_leave] = b_tab[i_leave] / pivot
        for i in range(m):
            if i == i_leave:
                continue
            factor = A_tab[i, j_enter]
            if abs(factor) > tol:
                A_tab[i, :] = A_tab[i, :] - factor * A_tab[i_leave, :]
                b_tab[i] = b_tab[i] - factor * b_tab[i_leave]

        basis[i_leave] = j_enter

    return None, A_tab, b_tab, basis, c[basis], c - c[basis] @ A_tab

In [29]:
def pricing_subproblem(pi, w, v, a_max, scale=100):
    """
    精确定价（二维有界整数背包 DP）：
    max pi @ a
    s.t. w @ a <= 1, v @ a <= 1, 0 <= a_i <= a_max_i, a_i 为整数

    返回:
      - 若存在改进列: (a, reduced_cost)
      - 否则: (None, None)
    其中 reduced_cost = 1 - pi @ a（对应最小化主问题）
    """
    pi = np.asarray(pi, dtype=float)
    w = np.asarray(w, dtype=float)
    v = np.asarray(v, dtype=float)
    a_max = np.asarray(a_max, dtype=int)
    n = len(pi)

    # 离散化容量（本例 w,v 为两位小数，scale=100 可精确表示）
    W = int(round(1.0 * scale))
    V = int(round(1.0 * scale))
    w_int = np.rint(w * scale).astype(int)
    v_int = np.rint(v * scale).astype(int)

    NEG = -1e30
    # dp[i][cw][cv]: 前 i 种物资、容量(cw,cv)的最大价值
    dp = np.full((n + 1, W + 1, V + 1), NEG, dtype=float)
    # choice 记录该状态在第 i 种物资上取了多少
    choice = np.zeros((n + 1, W + 1, V + 1), dtype=int)
    dp[0, 0, 0] = 0.0

    for i in range(1, n + 1):
        wi = w_int[i - 1]
        vi = v_int[i - 1]
        pim = pi[i - 1]
        ub = int(a_max[i - 1])
        for cw in range(W + 1):
            for cv in range(V + 1):
                best_val = NEG
                best_k = 0
                # 枚举第 i-1 种物资取 k 件
                max_k_by_cap = ub
                if wi > 0:
                    max_k_by_cap = min(max_k_by_cap, cw // wi)
                if vi > 0:
                    max_k_by_cap = min(max_k_by_cap, cv // vi)
                for k in range(max_k_by_cap + 1):
                    pw = cw - k * wi
                    pv = cv - k * vi
                    prev = dp[i - 1, pw, pv]
                    if prev <= NEG / 2:
                        continue
                    val = prev + k * pim
                    if val > best_val + 1e-12:
                        best_val = val
                        best_k = k
                dp[i, cw, cv] = best_val
                choice[i, cw, cv] = best_k

    # 在所有可行容量内找最大价值
    best_val = NEG
    best_w, best_v = 0, 0
    for cw in range(W + 1):
        for cv in range(V + 1):
            if dp[n, cw, cv] > best_val + 1e-12:
                best_val = dp[n, cw, cv]
                best_w, best_v = cw, cv

    if best_val <= 1.0 + 1e-10:
        return None, None

    # 回溯得到最优 a
    a = np.zeros(n, dtype=float)
    cw, cv = best_w, best_v
    for i in range(n, 0, -1):
        k = int(choice[i, cw, cv])
        a[i - 1] = k
        cw -= k * w_int[i - 1]
        cv -= k * v_int[i - 1]

    reduced_cost = 1.0 - float(best_val)
    return a, reduced_cost

In [57]:
def column_generation(A, b, c, basis, w, v, a_max, max_outer=20):
    """
    按“固定初始基”逻辑做列生成（最小化）：
    - 每轮先把当前有限主问题优化到最优单纯形表
    - 取初始基变量在该最优表中的列矩阵作为 B逆
    - pi = cB @ B逆
    - 定价求 p_j（由原约束生成的新列先转成初始基坐标）
    - 把 B逆 @ p_j 作为新列加入最优表，再继续单纯形优化
    """
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    c = np.asarray(c, dtype=float)

    m, n = A.shape
    basis0 = basis.copy()  # 记录初始基索引，不再改变

    # 初始基矩阵 B0（原问题坐标）及其逆：用于把定价得到的原列 a 转成 p = B0^{-1} a
    B0 = A[:, basis0].copy()
    B0_inv = np.linalg.inv(B0)

    # 把初始主问题转成“初始基表”（此时 basis0 列为单位阵）
    A_tab = B0_inv @ A
    b_tab = B0_inv @ b

    history = []
    last_step_added_column = False

    for outer in range(max_outer):
        last_step_added_column = False
        obj, A_tab, b_tab, basis, cB, sigma = simplex_optimize_tableau(
            A_tab, b_tab, c, basis
        )
        if obj is None:
            print("单纯形无界或失败")
            break

        # B逆：初始基变量在当前最优表中的列矩阵（你定义的 B逆）
        B_inv_user = A_tab[:, basis0]
        pi = cB @ B_inv_user

        # 定价：先用 pi 对应回原列 a 的收益系数
        pi_for_a = pi @ B0_inv
        a_new, reduced_dp = pricing_subproblem(pi_for_a, w, v, a_max)

        # 先创建该轮日志，再补充字段
        rec = {
            "iter": int(outer),
            "obj": float(obj),
            "n_cols_before": int(A_tab.shape[1]),
            "basis": [int(x) for x in basis],
            "a": None,
            "pi_a": None,
            "reduced_dp": None,
            "reduced_table": None,
            "added": False,
            "new_col_index": None,
            "stop_reason": None,
        }

        if a_new is None:
            rec["stop_reason"] = "no_pricing_column"
            history.append(rec)
            print(f"第 {outer+1} 轮后无改进列，已最优。目标值 = {obj:.6f}")
            break

        # p_j 是新列在“初始基坐标”中的表达
        p_j = B0_inv @ a_new
        reduced = 1.0 - pi @ p_j

        rec["a"] = a_new.astype(int).tolist()
        rec["pi_a"] = float(pi_for_a @ a_new)
        rec["reduced_dp"] = float(reduced_dp)
        rec["reduced_table"] = float(reduced)

        # 每轮输出：定价得到的新运输方案 a
        print(f"第 {outer+1} 轮定价: a = {rec['a']}")
        print(f"  pi_for_a @ a = {rec['pi_a']:.6f}, reduced(dp) = {rec['reduced_dp']:.6f}, reduced(table) = {rec['reduced_table']:.6f}")

        if reduced >= -1e-10:
            rec["stop_reason"] = "non_negative_reduced_cost"
            history.append(rec)
            print(f"第 {outer+1} 轮后无负检验数新列，已最优。目标值 = {obj:.6f}")
            break

        # 把 B逆 @ p_j 加入当前最优表（尚未入基）
        new_col_in_opt_tableau = (B_inv_user @ p_j).reshape(-1, 1)
        A_tab = np.hstack([A_tab, new_col_in_opt_tableau])
        c = np.append(c, 1.0)

        rec["added"] = True
        rec["new_col_index"] = int(A_tab.shape[1] - 1)
        rec["n_cols_after"] = int(A_tab.shape[1])
        history.append(rec)
        last_step_added_column = True

        print(
            f"  添加新列 j={A_tab.shape[1]-1}, reduced={reduced:.6f}, 当前列数={A_tab.shape[1]}"
        )

    # 如果最后一步是“加列后就退出循环”（例如达到 max_outer），补做一次单纯形优化
    if last_step_added_column:
        obj_post, A_tab, b_tab, basis, cB_post, sigma_post = simplex_optimize_tableau(
            A_tab, b_tab, c, basis
        )
        if obj_post is None:
            print("补充优化阶段出现无界或失败")
        else:
            history.append({
                "iter": int(len(history)),
                "obj": float(obj_post),
                "n_cols_before": int(A_tab.shape[1]),
                "basis": [int(x) for x in basis],
                "a": None,
                "pi_a": None,
                "reduced_dp": None,
                "reduced_table": None,
                "added": False,
                "new_col_index": None,
                "n_cols_after": int(A_tab.shape[1]),
                "stop_reason": "post_add_optimization",
            })

    # 由当前表恢复一个解向量
    x = np.zeros(A_tab.shape[1])
    for i, j in enumerate(basis):
        x[j] = b_tab[i]

    # 最终的“B逆”（按你的定义）：初始基在最终表中的列矩阵
    B_inv_user_final = A_tab[:, basis0]
    x_B_final = b_tab.copy()

    return A_tab, b_tab, c, basis, B_inv_user_final, x_B_final, history

In [58]:
# 用你已有的数据构建初始 RMP 并运行列生成
a_max_list = [min(int(1/w[i]), int(1/v[i])) for i in range(8)]
A0 = np.diag(a_max_list).astype(float)
A0 = np.hstack([A0, -np.eye(8)])
c0 = np.array([1.0] * 8 + [0.0] * 8)
b0 = np.array(q, dtype=float)
basis0 = list(range(8))

A_final, b_final, c_final, basis_final, B_inv_final, x_B_final, hist = column_generation(
    A0.copy(), b0, c0.copy(), basis0, w, v, a_max_list,max_outer=5
)
print("\n最终基变量索引:", basis_final)
print("最终基变量取值 x_B:", x_B_final)
print("最终目标值:", np.dot(c_final[basis_final], x_B_final))

第 1 轮定价: a = [0, 0, 2, 0, 0, 0, 2, 1]
  pi_for_a @ a = 1.542857, reduced(dp) = -0.542857, reduced(table) = -0.542857
  添加新列 j=16, reduced=-0.542857, 当前列数=17
第 2 轮定价: a = [0, 4, 0, 0, 1, 0, 0, 1]
  pi_for_a @ a = 1.276190, reduced(dp) = -0.276190, reduced(table) = -0.276190
  添加新列 j=17, reduced=-0.276190, 当前列数=18


/var/folders/_9/bwrp540d21j6tn6bbmtzrxww0000gn/T/ipykernel_5251/1316156433.py:33: RuntimeWarning: divide by zero encountered in divide
  theta = np.where(col > tol, b_tab / col, np.inf)


第 3 轮定价: a = [0, 0, 0, 0, 3, 0, 0, 1]
  pi_for_a @ a = 1.142857, reduced(dp) = -0.142857, reduced(table) = -0.142857
  添加新列 j=18, reduced=-0.142857, 当前列数=19
第 4 轮定价: a = [1, 0, 0, 0, 0, 0, 5, 0]
  pi_for_a @ a = 1.100000, reduced(dp) = -0.100000, reduced(table) = -0.100000
  添加新列 j=19, reduced=-0.100000, 当前列数=20
第 5 轮定价: a = [0, 0, 0, 0, 1, 3, 0, 0]
  pi_for_a @ a = 1.035714, reduced(dp) = -0.035714, reduced(table) = -0.035714
  添加新列 j=20, reduced=-0.035714, 当前列数=21

最终基变量索引: [0, 17, 16, 3, 18, 20, 19, 7]
最终基变量取值 x_B: [2.44000000e+00 5.00000000e+00 6.00000000e+00 7.66666667e+00
 4.44089210e-16 1.00000000e+01 5.60000000e+00 2.00000000e+00]
最终目标值: 38.70666666666666


In [59]:
for r in hist:
    print(
        f"iter={r['iter']}, obj={r['obj']:.6f}, cols={r['n_cols_before']}"
        + (f"->{r.get('n_cols_after')}" if r.get('n_cols_after') is not None else "")
        + f", added={r['added']}, new_col={r['new_col_index']}, stop={r['stop_reason']}"
    )
    if r['a'] is not None:
        print(
            f"  a={r['a']}, pi_a={r['pi_a']:.6f}, "
            f"reduced_dp={r['reduced_dp']:.6f}, reduced_table={r['reduced_table']:.6f}"
        )

iter=0, obj=44.738095, cols=16->17, added=True, new_col=16, stop=None
  a=[0, 0, 2, 0, 0, 0, 2, 1], pi_a=1.542857, reduced_dp=-0.542857, reduced_table=-0.542857
iter=1, obj=41.480952, cols=17->18, added=True, new_col=17, stop=None
  a=[0, 4, 0, 0, 1, 0, 0, 1], pi_a=1.276190, reduced_dp=-0.276190, reduced_table=-0.276190
iter=2, obj=40.100000, cols=18->19, added=True, new_col=18, stop=None
  a=[0, 0, 0, 0, 3, 0, 0, 1], pi_a=1.142857, reduced_dp=-0.142857, reduced_table=-0.142857
iter=3, obj=39.623810, cols=19->20, added=True, new_col=19, stop=None
  a=[1, 0, 0, 0, 0, 0, 5, 0], pi_a=1.100000, reduced_dp=-0.100000, reduced_table=-0.100000
iter=4, obj=39.063810, cols=20->21, added=True, new_col=20, stop=None
  a=[0, 0, 0, 0, 1, 3, 0, 0], pi_a=1.035714, reduced_dp=-0.035714, reduced_table=-0.035714
iter=5, obj=38.706667, cols=21->21, added=False, new_col=None, stop=post_add_optimization


In [60]:
print(A_final)

[[ 1.          0.          0.04        0.          0.          0.
  -0.1         0.         -0.1         0.         -0.02        0.
   0.          0.          0.02        0.          0.          0.
   0.          0.          0.        ]
 [ 0.          1.25        0.          0.          0.          0.
   0.          0.          0.         -0.25        0.          0.
   0.          0.          0.          0.          0.          1.
   0.          0.          0.        ]
 [ 0.          0.          1.          0.          0.          0.
   0.          0.          0.          0.         -0.5         0.
   0.          0.          0.          0.          1.          0.
   0.          0.          0.        ]
 [ 0.          0.          0.          1.          0.          0.
   0.          0.          0.          0.          0.         -0.33333333
   0.          0.          0.          0.          0.          0.
   0.          0.          0.        ]
 [ 0.         -0.41666667  0.          0.   

In [61]:
print(b_final)
print(c_final)
print(basis_final)
print(B_inv_final)




[2.44000000e+00 5.00000000e+00 6.00000000e+00 7.66666667e+00
 4.44089210e-16 1.00000000e+01 5.60000000e+00 2.00000000e+00]
[1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]
[0, 17, 16, 3, 18, 20, 19, 7]
[[ 1.          0.          0.04        0.          0.          0.
  -0.1         0.        ]
 [ 0.          1.25        0.          0.          0.          0.
   0.          0.        ]
 [ 0.          0.          1.          0.          0.          0.
   0.          0.        ]
 [ 0.          0.          0.          1.          0.          0.
   0.          0.        ]
 [ 0.         -0.41666667  0.          0.          1.         -0.44444444
   0.          0.        ]
 [ 0.          0.          0.          0.          0.          1.33333333
   0.          0.        ]
 [ 0.          0.         -0.4         0.          0.          0.
   1.          0.        ]
 [ 0.         -0.11904762 -0.14285714  0.         -0.14285714  0.06349206
   0.          1.        ]]


In [62]:
# 根据basis_final和x_B_final，得到x_final
x_final=np.zeros(A_final.shape[1])
for i in range(len(basis_final)):
    x_final[basis_final[i]]=x_B_final[i]
print(x_final)

[2.44000000e+00 0.00000000e+00 0.00000000e+00 7.66666667e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 2.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 6.00000000e+00 5.00000000e+00 4.44089210e-16 5.60000000e+00
 1.00000000e+01]


In [63]:
# 将x_final向上取整，得到整数规划近似解
x_final_int=np.ceil(x_final)
print(x_final_int)

# 计算整数规划近似解的目标值
obj_int=np.dot(c_final, x_final_int)
print(obj_int)


[ 3.  0.  0.  8.  0.  0.  0.  2.  0.  0.  0.  0.  0.  0.  0.  0.  6.  5.
  1.  6. 10.]
41.0
